# 🤝 Win / Loss de Oportunidades — Random Forest Classifier
> **Caso de negocio:** Tecsup / ICPNA · Equipo Comercial B2B  
> **Framework:** CRISP-DM · Digital Marketing Analytics — UPC  
> **Autor:** Miguel Salazar · Attach Group

---

## Contexto del problema

El equipo comercial tiene un **win rate del 28%** sobre ~350 oportunidades activas por trimestre. Los gerentes no saben qué deals priorizar: dedican el mismo tiempo a oportunidades calientes y a deals que ya están perdidos.

**Objetivo:** Predecir la probabilidad de cierre de cada oportunidad para que el equipo enfoque esfuerzos donde hay mayor chance de ganar.

**KPIs de éxito:**
- Win rate de 28% a 40%+
- Identificar deals en riesgo con 2 semanas de anticipación
- Reducir ciclo de venta promedio -20%

**Algoritmo:** Random Forest (ensemble de árboles paralelos con bagging)

## 📦 Fase 0 — Setup

In [ ]:
!pip install -q shap plotly

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
import shap
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Setup completo ✓')

## 1️⃣ Fase 1 — Business Understanding

In [ ]:
# Taxonomía del pipeline de ventas B2B
ETAPAS_PIPELINE = {
    'Prospección':      'Lead identificado, primer contacto realizado',
    'Calificación':     'Necesidad confirmada, presupuesto explorado',
    'Propuesta':        'Propuesta técnica y económica enviada',
    'Negociación':      'Condiciones en discusión',
    'Cierre':           'Ganado ✅ o Perdido ❌',
}

VARIABLES_MODELO = {
    'monto_oportunidad_k': 'Valor del deal en S/. miles — deals grandes → más competencia',
    'dias_ciclo':          'Días que lleva en pipeline — si se alarga, suele perderse',
    'n_reuniones':         'Nº de reuniones realizadas — proxy de interés real del cliente',
    'n_competidores':      'Competidores identificados — más competidores → menor win prob',
    'decision_makers':     'Nº de aprobadores — más = proceso más complejo',
    'propuesta_personalizada': 'Si se envió propuesta adaptada al cliente (0/1)',
}

print('Variables del modelo:')
for var, desc in VARIABLES_MODELO.items():
    print(f'  {var:30s} → {desc}')

## 2️⃣ Fase 2 — Data Understanding

In [ ]:
# Generación de datos sintéticos con estructura de CRM real
N = 1000

monto          = np.round(np.random.lognormal(4.5, 0.8, N)).clip(10, 500).astype(int)
dias_ciclo     = np.random.randint(7, 181, N)
n_reuniones    = np.random.randint(1, 11, N)
n_competidores = np.random.randint(0, 5, N)
decision_makers = np.random.randint(1, 6, N)
propuesta      = (np.random.random(N) > 0.45).astype(int)

# Proceso generador: relaciones basadas en experiencia comercial real
z = (-1.0
     + 0.004 * monto           # más monto → más motivación para ganar
     + 0.30  * n_reuniones     # más reuniones → mayor interés del cliente
     - 0.40  * n_competidores  # más competidores → menor probabilidad
     + 0.60  * propuesta       # propuesta personalizada → diferenciador clave
     + 0.15  * decision_makers # más DMs → señal de seriedad
     - 0.008 * dias_ciclo      # ciclos muy largos suelen perderse
     + np.random.normal(0, 0.9, N))

prob   = 1 / (1 + np.exp(-z))
ganado = (prob > 0.5).astype(int)

df = pd.DataFrame({
    'oportunidad_id':          range(1, N+1),
    'monto_oportunidad_k':     monto,
    'dias_ciclo':              dias_ciclo,
    'n_reuniones':             n_reuniones,
    'n_competidores':          n_competidores,
    'decision_makers':         decision_makers,
    'propuesta_personalizada': propuesta,
    'ganado':                  ganado
})

print(f'Dataset: {df.shape}')
wr = df['ganado'].mean()
print(f'Win rate actual: {wr:.1%}')
df.head()

In [ ]:
# Análisis: diferencias entre deals ganados y perdidos
comparacion = df.groupby('ganado')[['monto_oportunidad_k','dias_ciclo',
                                     'n_reuniones','n_competidores',
                                     'decision_makers']].mean().round(1)
comparacion.index = ['Perdido ❌', 'Ganado ✅']
print('=== DIFERENCIAS ENTRE GANADOS Y PERDIDOS ===')
display(comparacion)

In [ ]:
# Visualización: distribuciones por resultado
fig = make_subplots(rows=2, cols=3,
                    subplot_titles=['Monto (S/.k)','Días ciclo','Reuniones',
                                    'Competidores','Decision makers','Propuesta person.'])
features_viz = ['monto_oportunidad_k','dias_ciclo','n_reuniones',
                'n_competidores','decision_makers','propuesta_personalizada']
colors = {0: '#c0392b', 1: '#0d9488'}

for idx, feat in enumerate(features_viz):
    r, c = idx // 3 + 1, idx % 3 + 1
    for cls, label in [(0,'Perdido'), (1,'Ganado')]:
        data = df[df['ganado'] == cls][feat]
        fig.add_trace(
            go.Histogram(x=data, name=label, opacity=0.65,
                         marker_color=colors[cls],
                         showlegend=(idx == 0)),
            row=r, col=c
        )

fig.update_layout(barmode='overlay', height=500,
                  title='Distribución de variables por resultado (Ganado vs Perdido)',
                  plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Win rate por número de reuniones y propuesta personalizada
pivot = df.groupby(['n_reuniones','propuesta_personalizada'])['ganado'].mean().unstack()
pivot.columns = ['Sin propuesta pers.', 'Con propuesta pers.']

fig = px.line(pivot.reset_index(), x='n_reuniones',
              y=['Sin propuesta pers.', 'Con propuesta pers.'],
              title='Win rate por reuniones y propuesta personalizada',
              markers=True)
fig.update_yaxes(tickformat='.0%', title='Win rate')
fig.update_xaxes(title='Número de reuniones')
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white', height=380)
fig.show()

## 3️⃣ Fase 3 — Data Preparation

In [ ]:
FEATURES = ['monto_oportunidad_k', 'dias_ciclo', 'n_reuniones',
            'n_competidores', 'decision_makers', 'propuesta_personalizada']
TARGET   = 'ganado'

X = df[FEATURES].copy()
y = df[TARGET].astype(int)

# Feature engineering: señales derivadas
X['ratio_reuniones_ciclo'] = X['n_reuniones'] / X['dias_ciclo']  # velocidad de avance
X['flag_alta_competencia'] = (X['n_competidores'] >= 3).astype(int)  # escenario muy competido
X['log_monto'] = np.log1p(X['monto_oportunidad_k'])  # normaliza distribución log-normal

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Win rate train: {y_train.mean():.1%} | Win rate test: {y_test.mean():.1%}')

## 4️⃣ Fase 4 — Modeling

In [ ]:
# Random Forest: ensemble de árboles independientes con votación mayoritaria
# class_weight='balanced': compensa el desbalance (más perdidos que ganados)
rf = RandomForestClassifier(
    n_estimators=200,       # número de árboles
    max_depth=6,            # profundidad — evita overfitting
    min_samples_leaf=5,     # mínimo de muestras en hoja
    class_weight='balanced',# penaliza más los errores en la clase minoritaria
    random_state=42,
    n_jobs=-1               # paraleliza en todos los cores
)

rf.fit(X_train, y_train)

# Validación cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(rf, X_train, y_train, cv=cv, scoring='roc_auc')
cv_rec = cross_val_score(rf, X_train, y_train, cv=cv, scoring='recall')

print(f'CV AUC-ROC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}')
print(f'CV Recall:  {cv_rec.mean():.3f} ± {cv_rec.std():.3f}')

In [ ]:
# Importancia de variables
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

fi = pd.DataFrame({'feature': X_train.columns,
                   'importance': rf.feature_importances_}
                 ).sort_values('importance', ascending=True)

fig = go.Figure(go.Bar(
    x=fi['importance'], y=fi['feature'], orientation='h',
    marker_color='#1a4c8c',
    text=fi['importance'].map('{:.3f}'.format),
    textposition='outside'
))
fig.update_layout(title='Importancia de variables — Random Forest',
                  height=400, plot_bgcolor='white', paper_bgcolor='white',
                  xaxis=dict(gridcolor='#f0f0f0'),
                  yaxis=dict(showgrid=False))
fig.show()

## 5️⃣ Fase 5 — Evaluation

In [ ]:
# Métricas en test set
metrics = {
    'Accuracy':  accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall':    recall_score(y_test, y_pred, zero_division=0),
    'F1-Score':  f1_score(y_test, y_pred, zero_division=0),
    'AUC-ROC':   roc_auc_score(y_test, y_prob),
}
criterios = {'AUC-ROC': 0.75, 'Recall': 0.68, 'Precision': 0.62}

print('=== MÉTRICAS EN TEST SET ===')
for k, v in metrics.items():
    umbral = criterios.get(k)
    estado = ''
    if umbral:
        estado = '✅ APROBADO' if v >= umbral else f'❌ (umbral {umbral:.2f})'
    print(f'  {k:12s}: {v:.4f}  {estado}')

In [ ]:
# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                          name=f'RF (AUC={auc:.3f})',
                          line=dict(color='#1a4c8c', width=2.5)))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
                          name='Random', line=dict(color='#ccc', dash='dash')))
fig.update_layout(title='Curva ROC — Win/Loss de Oportunidades',
                  xaxis_title='FPR', yaxis_title='TPR (Recall)',
                  height=400, plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=['Perdido', 'Ganado'])
disp.plot(ax=ax, colorbar=False, cmap='Greens')
ax.set_title('Matriz de Confusión')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'\nFN (deals ganados que perdemos de atender): {fn}')
print(f'FP (tiempo invertido en deals perdidos):    {fp}')
print(f'VP (deals ganados correctamente priorizados): {tp}')

In [ ]:
# Análisis de umbral óptimo: el modelo no tiene que usar 0.5 por defecto
thresholds = np.arange(0.1, 0.9, 0.05)
resultados = []
for t in thresholds:
    preds = (y_prob >= t).astype(int)
    prec = precision_score(y_test, preds, zero_division=0)
    rec  = recall_score(y_test, preds, zero_division=0)
    f1   = f1_score(y_test, preds, zero_division=0)
    resultados.append({'umbral': t, 'precision': prec, 'recall': rec, 'f1': f1})

df_thresh = pd.DataFrame(resultados)
umbral_optimo = df_thresh.loc[df_thresh['f1'].idxmax(), 'umbral']

fig = go.Figure()
for col, color in [('precision','#c0392b'), ('recall','#0d9488'), ('f1','#1a4c8c')]:
    fig.add_trace(go.Scatter(x=df_thresh['umbral'], y=df_thresh[col],
                              mode='lines+markers', name=col, line=dict(color=color)))
fig.add_vline(x=umbral_optimo, line_dash='dash', line_color='orange',
              annotation_text=f'Umbral óptimo F1: {umbral_optimo:.2f}')
fig.update_layout(title='Precision / Recall / F1 por umbral de decisión',
                  xaxis_title='Umbral', height=380,
                  plot_bgcolor='white', paper_bgcolor='white')
fig.show()
print(f'Umbral óptimo (max F1): {umbral_optimo:.2f}')

In [ ]:
# SHAP: por qué el modelo gana o pierde un deal específico
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# Waterfall del deal con mayor probabilidad de ganar
mejor_deal_idx = y_prob.argmax()
print(f'Deal con mayor probabilidad de ganar: {y_prob[mejor_deal_idx]:.1%}')
print(df.iloc[X_test.index[mejor_deal_idx]][FEATURES + ['ganado']])

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[1][mejor_deal_idx],
        base_values=explainer.expected_value[1],
        data=X_test.iloc[mejor_deal_idx],
        feature_names=X_test.columns.tolist()
    )
)

## 6️⃣ Fase 6 — Deployment

In [ ]:
# Aplicar modelo a todo el pipeline activo
X_all = df[FEATURES].copy()
X_all['ratio_reuniones_ciclo'] = X_all['n_reuniones'] / X_all['dias_ciclo']
X_all['flag_alta_competencia'] = (X_all['n_competidores'] >= 3).astype(int)
X_all['log_monto'] = np.log1p(X_all['monto_oportunidad_k'])

df['win_probability'] = (rf.predict_proba(X_all)[:, 1] * 100).round(1)
df['prioridad'] = pd.cut(
    df['win_probability'],
    bins=[0, 35, 65, 100],
    labels=['Deal frío ❌', 'Deal en riesgo ⚠️', 'Deal caliente ✅']
)

print('=== PIPELINE PRIORIZADO ===')
print(df['prioridad'].value_counts())
print(f'\nRevenue en deals calientes: S/. {df[df["prioridad"]=="Deal caliente ✅"]["monto_oportunidad_k"].sum():,.0f}K')

# Top 10 deals a priorizar
df[['oportunidad_id','monto_oportunidad_k','win_probability','prioridad']]\
    .sort_values('win_probability', ascending=False)\
    .head(10)

In [ ]:
# Guardar outputs
import joblib
df[['oportunidad_id','win_probability','prioridad']].to_csv('pipeline_priorizado.csv', index=False)
joblib.dump(rf, 'modelo_win_loss.pkl')
print('Outputs guardados ✓')

print('\n=== RESUMEN EJECUTIVO ===')
n_calientes = (df['prioridad'] == 'Deal caliente ✅').sum()
n_total = len(df)
rev_calientes = df[df['prioridad']=='Deal caliente ✅']['monto_oportunidad_k'].sum()
rev_total = df['monto_oportunidad_k'].sum()
print(f'Deals calientes (>65%):  {n_calientes} de {n_total} ({n_calientes/n_total:.0%})')
print(f'Revenue en calientes:    S/.{rev_calientes:,.0f}K de S/.{rev_total:,.0f}K ({rev_calientes/rev_total:.0%})')
print(f'\nArquitectura de producción:')
print('  CRM Plugin → Cloud Run API → score en tiempo real → alerta al gerente comercial')